In [1]:
# ============================================================
# FUNCTION 2 - WEEK 13 FINAL-WEEK BAYESIAN OPTIMISATION
# Run from inside week13/
# ============================================================
#
# FINAL-WEEK STRATEGY
#
# Recent evidence:
#
# incumbent:
# [0.711177, 1.000000] -> 0.6796627330
#
# Week 10:
# [0.702316, 0.995076] -> 0.6682000916
#
# Week 11:
# [0.702314, 0.972050] -> 0.5895778573
#
# Week 12:
# [0.721178, 1.000000] -> 0.5854534703
#
# This suggests:
# - x2 should remain very close to 1
# - increasing x1 beyond the incumbent was harmful
# - the remaining useful direction is toward lower x1
#
# Since this is the FINAL query:
# - prioritise exploitation
# - use highest mean + low-beta UCB as primary evidence
# - keep EI as a diagnostic only
# - explicitly search x2=1
# - do not chase high uncertainty
# ============================================================


# ------------------------------------------------------------
# 0. Limit numerical-library threading
# ------------------------------------------------------------

import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"


# ------------------------------------------------------------
# Imports
# ------------------------------------------------------------

import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)


# ------------------------------------------------------------
# 1. Load Week 13 cumulative data
# ------------------------------------------------------------

X = np.load(
    "function2/initial_inputs.npy"
)

Y = np.load(
    "function2/initial_outputs.npy"
).reshape(-1)

best_idx = np.argmax(Y)

best_x = X[best_idx]
best_y = Y[best_idx]

tree = cKDTree(X)

print("================================")
print("DATA")
print("================================")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(
    best_x,
    "->",
    best_y
)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. Week 12 calibration check
# ------------------------------------------------------------
#
# Week 12 final candidate:
# [0.721178, 1.000000]
#
# Its GP prediction should be recomputed from the Week 12
# model if exact notebook values are required.
#
# Here the realised result itself is the main diagnostic:
# moving x1 upward by ~0.010 reduced performance substantially.
# ------------------------------------------------------------

week12_x = np.array([
    0.721178,
    1.000000
])

week12_actual = (
    0.585453470321145
)

print("\n================================")
print("WEEK 12 RESULT")
print("================================")

print(
    "Week 12 input:",
    week12_x
)

print(
    "Week 12 actual:",
    week12_actual
)

print(
    "\nDifference from incumbent:"
)

print(
    week12_actual
    - best_y
)

print(
    "\nx1 displacement from incumbent:"
)

print(
    week12_x[0]
    - best_x[0]
)


# ------------------------------------------------------------
# 3. Recent local evidence
# ------------------------------------------------------------

week10_x = np.array([
    0.702316,
    0.995076
])

week10_y = (
    0.6682000915555967
)

week11_x = np.array([
    0.702314,
    0.972050
])

week11_y = (
    0.5895778573315174
)

print("\n================================")
print("RECENT LOCAL EVIDENCE")
print("================================")

print("\nIncumbent:")
print(
    best_x,
    "->",
    best_y
)

print("\nWeek 10:")
print(
    week10_x,
    "->",
    week10_y
)

print("\nWeek 11:")
print(
    week11_x,
    "->",
    week11_y
)

print("\nWeek 12:")
print(
    week12_x,
    "->",
    week12_actual
)


# ------------------------------------------------------------
# 4. Fit updated ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(
            1e-3,
            1e3
        )
    )
    *
    Matern(
        length_scale=np.ones(2) * 0.2,
        length_scale_bounds=(
            0.01,
            2.0
        ),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(
            1e-8,
            1e-1
        )
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,

    # Reduced from 30 because this is the final local fit
    # and it also lowers thread/resource pressure.
    n_restarts_optimizer=15,

    random_state=42
)

gp.fit(
    X,
    Y
)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(
    gp.kernel_
)

lengthscales = (
    gp.kernel_.k1.k2.length_scale
)

inverse_ls = (
    1.0
    / lengthscales
)

relative_sensitivity = (
    inverse_ls
    / inverse_ls.sum()
)

print(
    "\nARD lengthscales:"
)
print(
    lengthscales
)

print(
    "\nNormalised inverse-lengthscale sensitivity:"
)
print(
    relative_sensitivity
)


# ------------------------------------------------------------
# 5. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = (
        mu
        - best_y
        - xi
    )

    valid = (
        sigma > 1e-12
    )

    Z = np.zeros_like(mu)

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 6. Empirical local scale
# ------------------------------------------------------------

other_mask = (
    np.arange(len(X))
    != best_idx
)

distances_to_best = np.linalg.norm(
    X[other_mask]
    - best_x,
    axis=1
)

nearest_distance = (
    distances_to_best.min()
)

# We need slightly more room than Week 12 because the immediate
# lower-x1 side is already occupied by the strong Week 10 point.
#
# This is still a local final-week search.

empirical_cap = min(
    2.0 * nearest_distance,
    0.03
)

print("\n================================")
print("EMPIRICAL LOCAL SCALE")
print("================================")

print(
    "Nearest observed point:",
    nearest_distance
)

print(
    "\nEmpirical cap:",
    empirical_cap
)


# ------------------------------------------------------------
# 7. ARD-informed trust region
# ------------------------------------------------------------
#
# Give x1 enough room to search beyond the already-observed
# Week 10 neighbourhood.
# ------------------------------------------------------------

trust_half_width = np.clip(
    0.30 * lengthscales,
    0.015,
    empirical_cap
)

lower = np.maximum(
    0.0,
    best_x
    - trust_half_width
)

upper = np.minimum(
    1.0,
    best_x
    + trust_half_width
)

print("\n================================")
print("WEEK 13 TRUST REGION")
print("================================")

print("Centre:")
print(
    best_x
)

print(
    "\nHalf-widths:"
)
print(
    trust_half_width
)

print(
    "\nLower:"
)
print(
    lower
)

print(
    "\nUpper:"
)
print(
    upper
)


# ------------------------------------------------------------
# 8. Dense local candidate pool
# ------------------------------------------------------------

rng = np.random.default_rng(
    42
)

local_candidates = rng.uniform(
    lower,
    upper,
    size=(
        400000,
        2
    )
)


# ------------------------------------------------------------
# 9. Explicit x2 = 1 boundary profile
# ------------------------------------------------------------

x1_boundary_values = np.linspace(
    lower[0],
    upper[0],
    100000
)

boundary_candidates = np.column_stack([
    x1_boundary_values,
    np.ones_like(
        x1_boundary_values
    )
])


# ------------------------------------------------------------
# 10. Explicit x1 profile through incumbent x2
# ------------------------------------------------------------

x1_profile_candidates = np.column_stack([
    x1_boundary_values,
    np.full_like(
        x1_boundary_values,
        best_x[1]
    )
])


# ------------------------------------------------------------
# 11. Combine candidates and portal-round
# ------------------------------------------------------------

candidates = np.vstack([
    local_candidates,
    boundary_candidates,
    x1_profile_candidates
])

candidates = np.round(
    candidates,
    6
)


# ------------------------------------------------------------
# 12. Near-duplicate filtering
# ------------------------------------------------------------

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print(
    "\nCandidates after portal rounding + duplicate filtering:"
)

print(
    len(candidates)
)


# ------------------------------------------------------------
# 13. GP predictions
# ------------------------------------------------------------

mu, sigma = gp.predict(
    candidates,
    return_std=True
)


# ------------------------------------------------------------
# 14. EI - diagnostic only
# ------------------------------------------------------------

EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(
    EI
)

print("\n================================")
print("PRIMARY EI - DIAGNOSTIC ONLY")
print("================================")

print(
    "candidate =",
    candidates[ei_idx]
)

print(
    "mean =",
    mu[ei_idx]
)

print(
    "std =",
    sigma[ei_idx]
)

print(
    "EI =",
    EI[ei_idx]
)

print(
    "distance from incumbent =",
    np.linalg.norm(
        candidates[ei_idx]
        - best_x
    )
)


# ------------------------------------------------------------
# 15. Highest posterior mean
# ------------------------------------------------------------

mean_idx = np.argmax(
    mu
)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print(
    "candidate =",
    candidates[mean_idx]
)

print(
    "mean =",
    mu[mean_idx]
)

print(
    "std =",
    sigma[mean_idx]
)

print(
    "distance from incumbent =",
    np.linalg.norm(
        candidates[mean_idx]
        - best_x
    )
)


# ------------------------------------------------------------
# 16. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================")

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta
        * sigma
    )

    idx = np.argmax(
        UCB
    )

    print(
        f"\nbeta={beta}"
    )

    print(
        "candidate =",
        candidates[idx]
    )

    print(
        "mean =",
        mu[idx]
    )

    print(
        "std =",
        sigma[idx]
    )

    print(
        "distance =",
        np.linalg.norm(
            candidates[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 17. Trust-region boundary check
# ------------------------------------------------------------

def boundary_status(
    x,
    lower,
    upper,
    tol=0.001
):

    status = []

    for j in range(
        len(x)
    ):

        if abs(
            x[j]
            - lower[j]
        ) <= tol:

            status.append(
                f"x{j+1}=LOWER"
            )

        elif abs(
            x[j]
            - upper[j]
        ) <= tol:

            status.append(
                f"x{j+1}=UPPER"
            )

    if not status:
        return "interior"

    return ", ".join(
        status
    )


print("\n================================")
print("BOUNDARY CHECK")
print("================================")

print(
    "EI:",
    boundary_status(
        candidates[ei_idx],
        lower,
        upper
    )
)

print(
    "Highest mean:",
    boundary_status(
        candidates[mean_idx],
        lower,
        upper
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta
        * sigma
    )

    idx = np.argmax(
        UCB
    )

    print(
        f"UCB beta={beta}:",
        boundary_status(
            candidates[idx],
            lower,
            upper
        )
    )


# ------------------------------------------------------------
# 18. x2 = 1 boundary profile
# ------------------------------------------------------------

boundary_candidates = np.round(
    boundary_candidates,
    6
)

boundary_distance, _ = tree.query(
    boundary_candidates,
    k=1
)

boundary_valid = boundary_candidates[
    boundary_distance > 0.01
]

print("\n================================")
print("x2 = 1 BOUNDARY PROFILE")
print("================================")

if len(boundary_valid) > 0:

    boundary_mu, boundary_sigma = gp.predict(
        boundary_valid,
        return_std=True
    )

    boundary_idx = np.argmax(
        boundary_mu
    )

    print(
        "Best boundary candidate =",
        boundary_valid[boundary_idx]
    )

    print(
        "mean =",
        boundary_mu[boundary_idx]
    )

    print(
        "std =",
        boundary_sigma[boundary_idx]
    )

    print(
        "distance from incumbent =",
        np.linalg.norm(
            boundary_valid[boundary_idx]
            - best_x
        )
    )

else:

    print(
        "No admissible x2=1 points."
    )


# ------------------------------------------------------------
# 19. Week 12 mirror diagnostic
# ------------------------------------------------------------
#
# Week 12 moved upward in x1.
#
# Mirror the same displacement downward.
# This may be invalid because Week 10 already occupies that
# neighbourhood, which is itself informative.
# ------------------------------------------------------------

week12_step = (
    week12_x
    - best_x
)

mirror_candidate = (
    best_x
    - week12_step
)

mirror_candidate = np.clip(
    mirror_candidate,
    0.0,
    1.0
)

mirror_candidate = np.round(
    mirror_candidate,
    6
)

mirror_dist, mirror_idx = tree.query(
    mirror_candidate.reshape(1, -1),
    k=1
)

print("\n================================")
print("WEEK 12 MIRROR DIAGNOSTIC")
print("================================")

print(
    "Mirror candidate =",
    mirror_candidate
)

print(
    "Nearest observed distance =",
    mirror_dist[0]
)

print(
    "Nearest observed point =",
    X[mirror_idx[0]]
)

if mirror_dist[0] > 0.01:

    mirror_mu, mirror_sigma = gp.predict(
        mirror_candidate.reshape(1, -1),
        return_std=True
    )

    print(
        "mean =",
        mirror_mu[0]
    )

    print(
        "std =",
        mirror_sigma[0]
    )

else:

    print(
        "Mirror candidate is too close to an existing observation."
    )


# ------------------------------------------------------------
# 20. Best admissible lower-x1 candidate at x2=1
# ------------------------------------------------------------
#
# Final-week directional diagnostic:
# consider ONLY x1 values below the incumbent while holding
# x2 at 1.
# ------------------------------------------------------------

lower_side = boundary_valid[
    boundary_valid[:, 0]
    < best_x[0]
]

print("\n================================")
print("LOWER-x1 FINAL-WEEK DIAGNOSTIC")
print("================================")

if len(lower_side) > 0:

    lower_mu, lower_sigma = gp.predict(
        lower_side,
        return_std=True
    )

    lower_idx = np.argmax(
        lower_mu
    )

    print(
        "Best lower-x1 candidate =",
        lower_side[lower_idx]
    )

    print(
        "mean =",
        lower_mu[lower_idx]
    )

    print(
        "std =",
        lower_sigma[lower_idx]
    )

    print(
        "distance from incumbent =",
        np.linalg.norm(
            lower_side[lower_idx]
            - best_x
        )
    )

else:

    print(
        "No admissible lower-x1 candidates."
    )

DATA
X shape: (22, 2)
Y shape: (22,)

Current best:
[0.711177 1.      ] -> 0.679662733033218

Y range:
min = -0.06562362443733738
max = 0.679662733033218
std = 0.24857925422988106

WEEK 12 RESULT
Week 12 input: [0.721178 1.      ]
Week 12 actual: 0.585453470321145

Difference from incumbent:
-0.09420926271207297

x1 displacement from incumbent:
0.010001000000000038

RECENT LOCAL EVIDENCE

Incumbent:
[0.711177 1.      ] -> 0.679662733033218

Week 10:
[0.702316 0.995076] -> 0.6682000915555967

Week 11:
[0.702314 0.97205 ] -> 0.5895778573315174

Week 12:
[0.721178 1.      ] -> 0.585453470321145

GP FIT

Fitted kernel:
1.13**2 * Matern(length_scale=[0.0648, 2], nu=2.5) + WhiteKernel(noise_level=0.0369)

ARD lengthscales:
[0.06483785 1.99836047]

Normalised inverse-lengthscale sensitivity:
[0.96857411 0.03142589]

EMPIRICAL LOCAL SCALE
Nearest observed point: 0.010001000000000038

Empirical cap: 0.020002000000000075

WEEK 13 TRUST REGION
Centre:
[0.711177 1.      ]

Half-widths:
[0.01945135

In [2]:
# ============================================================
# FINAL FUNCTION 2 - WEEK 13 SELECTION
# ============================================================
#
# FINAL CAPSTONE QUERY
#
# Evidence:
# - Week 12 showed that increasing x1 from the incumbent was
#   harmful.
# - Week 11 showed that moving x2 substantially below 1 was
#   also harmful.
# - Week 10 remained strong with x2 very close to 1.
# - The updated GP prefers lower x1.
# - The explicit x2=1 boundary profile identifies an
#   admissible lower-x1 candidate.
#
# Although the unrestricted EI / mean / UCB candidate moves
# slightly inward in x2, the empirical observations provide
# stronger evidence for keeping x2 at the boundary.
#
# Therefore use a final controlled exploitation move:
# lower x1 while keeping x2 exactly at 1.
# ============================================================

week13_candidate = np.array([
    0.693612,
    1.000000
])

print("================================")
print("FUNCTION 2 - WEEK 13 FINAL")
print("================================")

print("\nCandidate:")
print(week13_candidate)

# ------------------------------------------------------------
# GP prediction
# ------------------------------------------------------------

final_mu, final_sigma = gp.predict(
    week13_candidate.reshape(1, -1),
    return_std=True
)

print("\nPredicted mean:")
print(final_mu[0])

print("\nPredicted std:")
print(final_sigma[0])


# ------------------------------------------------------------
# Distance from incumbent
# ------------------------------------------------------------

print("\nDistance from current best:")
print(
    np.linalg.norm(
        week13_candidate - best_x
    )
)


# ------------------------------------------------------------
# Check against all previous observations
# ------------------------------------------------------------

nearest_dist, nearest_idx = tree.query(
    week13_candidate.reshape(1, -1),
    k=1
)

print("\nNearest observed-point distance:")
print(nearest_dist[0])

print("\nNearest observed point:")
print(X[nearest_idx[0]])

print("\nDuplicate rule satisfied:")
print(
    nearest_dist[0] > 0.01
)


# ------------------------------------------------------------
# Portal format
# ------------------------------------------------------------

portal = "-".join(
    f"{x:.6f}"
    for x in week13_candidate
)

print("\nPortal format:")
print(portal)

FUNCTION 2 - WEEK 13 FINAL

Candidate:
[0.693612 1.      ]

Predicted mean:
0.6015974404764981

Predicted std:
0.0600595751807487

Distance from current best:
0.017564999999999942

Nearest observed-point distance:
0.010000269596365949

Nearest observed point:
[0.702316 0.995076]

Duplicate rule satisfied:
True

Portal format:
0.693612-1.000000
